# Bayesian HRF + AR(P) noise model, one image-space parcel (mnt-correct, Gaussian kernels)

This notebook summarizes [`run_Bayes_image-space_gaussian-kernel_mnt-correct.py`](run_Bayes_image-space_gaussian-kernel_mnt-correct.py). It contains no code to run. It covers:

1. [Overview](#1.-Overview)
2. [Model equations](#2.-Model-equations)
3. [Notation](#3.-Notation)
4. [Parameters: learnable and deterministic](#4.-Parameters)
5. [Step-by-step pipeline](#5.-Step-by-step-pipeline)
6. [Outputs](#6.-Outputs)
7. [Settings reference](#7.-Settings-reference)

The script fits the evoked HbO response to **correct mountain trials (mnt-correct)** of gradCPT in **one cortical parcel** of one subject. It uses a single Bayesian model in which three parts are estimated jointly with NUTS (PyMC):

- the HRF, as weights on 20 delayed Gaussian kernels,
- the nuisance regressors (per-run Legendre drift and per-run global signal),
- an AR(P) noise process with Student-t innovations.

Unlike the 3-stage GLM in `run_model_cont_EEG_fNIRS.py` (OLS drift → OLS GSR → AR-IRLS), nothing is regressed out beforehand. Uncertainty in the noise model therefore carries through to the HRF.

## 1. Overview

| | |
|---|---|
| **Data** | Image-space HbO time series of one parcel (`select_parcel`, e.g. `SomMotA_25_LH`), sensitive parcels only |
| **Events** | gradCPT mountain trials with `response_code == 0` (correct withholds) |
| **Event basis** | 20 Gaussian kernels centred at −2, −1, …, 17 s after onset (cedalion `GaussianKernels`) |
| **Nuisance** | Per-run Legendre drift (orders 0–3; order 0 is the run intercept) and per-run global signal (GSR) |
| **Noise** | Stationary AR(P) with Student-t innovations; P chosen from the OLS residuals (10 ≤ P ≤ 30) |
| **Inference** | NUTS, 4 chains × 500 draws (500 tuning), `target_accept = 0.9`; CPU (PyMC) or GPU (numpyro) |
| **Units** | Fitting is done on standardized data; results are converted back to µM |

## 2. Model equations

### 2.1 Observation model

$$
\tilde{y} = X\,h + N\,b + e, \qquad \tilde{y} = \frac{y - \bar{y}}{s_y}
$$

The concatenated, standardized parcel time series $\tilde y \in \mathbb{R}^{T}$ is the evoked part, plus the nuisance part, plus autocorrelated noise. In the earlier notation, $x = E\,\beta + e$ with $E = [X,\;N]$ and $\beta = [h;\,b]$.

### 2.2 Event regressors and the HRF

For kernel $j$ with centre $\mu_j = -2 + j$ s ($j = 0,\dots,L-1$, $L = 20$), cedalion defines

$$
g_j(\tau) = \exp\!\left(-\frac{(\tau-\mu_j)^2}{(2\,t_{std})^2}\right), \qquad \tau \in [-t_{pre},\, t_{post}] = [-2,\,18]\ \text{s},
$$

normalized to a peak of 1. With $t_{std} = 1$ s, the effective Gaussian SD is $\sqrt{2}\approx 1.41$ s.

Column $j$ of $X$ is the onset train convolved with kernel $j$:

$$
X_{t j} = \sum_{i \in \text{trials}} g_j(t - o_i).
$$

The HRF on the lag axis is a deterministic function of $h$:

$$
\text{hrf}(\tau) = \sum_j g_j(\tau)\,h_j \quad\Longleftrightarrow\quad \text{hrf} = B\,h .
$$

### 2.3 Noise model: AR(P) with Student-t innovations

$$
e_t = \sum_{k=1}^{P} \phi_k\, e_{t-k} + \varepsilon_t, \qquad \varepsilon_t \sim \text{StudentT}(\nu,\,0,\,\sigma).
$$

**Stationarity by construction.** The sampler works on unconstrained $z_k$. These map to partial autocorrelations $r_k = \tanh(z_k) \in (-1, 1)$, which map to $\phi$ through the Durbin–Levinson recursion:

$$
\phi^{(1)} = [r_1], \qquad \phi^{(k)} = \big[\phi^{(k-1)} - r_k\,\text{rev}(\phi^{(k-1)}),\; r_k\big], \qquad \phi = \phi^{(P)} .
$$

### 2.4 Likelihood (conditional, within-run)

Write $\mu = Xh + Nb$ and $e = \tilde y - \mu$. The innovations are

$$
\varepsilon_t = e_t - \sum_{k=1}^{P}\phi_k\, e_{t-k}, \qquad t \in \mathcal{T}_P ,
$$

where $\mathcal{T}_P$ holds the time points whose full $P$-lag history lies **inside the same run**. The first $P$ samples of each run are conditioned on, and no lag crosses a run boundary.

The map $e \mapsto \varepsilon$ is unit lower-triangular, so its Jacobian is 1. The exact conditional log-likelihood is therefore

$$
\log p(\tilde y \mid h, b, \phi, \sigma, \nu) = \sum_{t\in\mathcal{T}_P} \log \text{StudentT}(\varepsilon_t \mid \nu, 0, \sigma).
$$

In code this is added with `pm.Potential("lik", ...)`.

### 2.5 Priors

$$
\begin{aligned}
\tau_h &\sim \text{HalfNormal}(1) \\
h_j \mid \tau_h &\sim \mathcal{N}(0,\, \tau_h), \quad j = 1..L \\
\tau_s &\sim \text{HalfNormal}(0.5) \quad\text{(only if \texttt{is\_smooth\_penalty})}\\
\Delta^2 h_i \mid \tau_s &\sim \mathcal{N}(0,\, \tau_s), \quad i = 1..L-2 \quad\text{(penalty, \texttt{pm.Potential("smooth")})}\\
b_m &\sim \mathcal{N}(0,\, 1), \quad m = 1..M \\
z_k &\sim \mathcal{N}(0,\, 1/k), \quad k = 1..P \\
\sigma &\sim \text{HalfNormal}(1) \\
\nu - 2 &\sim \text{Gamma}(\alpha = 2,\ \beta = 0.1) \quad(\text{mean } 20)
\end{aligned}
$$

Here $\Delta^2 h_i = h_{i+2} - 2h_{i+1} + h_i$. With the penalty on, the effective prior on $h$ is

$$
p(h \mid \tau_h, \tau_s) \;\propto\; \prod_j \mathcal{N}(h_j \mid 0, \tau_h)\;\prod_i \mathcal{N}(\Delta^2 h_i \mid 0, \tau_s).
$$

There is **no canonical-HRF prior** and **no separate amplitude**. $h$ holds the kernel weights directly, so its prior is zero-mean and does not assume an HRF shape.

### 2.6 Posterior

$$
p(h, b, z, \sigma, \nu, \tau_h, \tau_s \mid \tilde y) \;\propto\; p(\tilde y \mid h, b, \phi(z), \sigma, \nu)\; p(h\mid\tau_h,\tau_s)\,p(\tau_h)\,p(\tau_s)\,p(b)\,p(z)\,p(\sigma)\,p(\nu).
$$

## 3. Notation

| Symbol | Code name | Size | Meaning |
|---|---|---|---|
| $y$ | `y` | $T$ | Parcel HbO in µM, runs concatenated |
| $\tilde y$ | `y_s` | $T$ | Standardized data, $(y-\bar y)/s_y$ |
| $\bar y,\ s_y$ | `y_mean`, `scale` | scalar | Mean and SD of $y$, used to convert back to µM |
| $T$ | `T` | scalar | Number of samples after truncation and concatenation |
| $R$ | `len(run_keys)` | scalar | Number of runs |
| $L$ | `B.shape[1]` | scalar | Number of Gaussian kernels (20) |
| $M$ | `N.shape[1]` | scalar | Number of nuisance regressors, $(\text{DRIFT\_ORDER}+2)\cdot R = 5R$ |
| $P$ | `P` | scalar | AR order, fixed before sampling |
| $o_i$ | `mnt_onsets` | | Onset of mnt-correct trial $i$ |
| $\tau$ | `hrf_t` | | Lag axis of the HRF, −2 … 18 s |
| $g_j(\tau)$, $B$ | `B` (`gauss_basis`) | $n_\tau \times L$ | Gaussian kernels on the lag axis |
| $X$ | `X_ev` | $T \times L$ | Event regressors (onsets convolved with each kernel) |
| $N$ | `N` | $T \times M$ | Nuisance regressors: `Drift 0..3 run r`, `GS run r` |
| $h$ | `h` | $L$ | Kernel weights (HRF coefficients) |
| $b$ | `b` | $M$ | Nuisance weights |
| $\mu$ | `mu` | $T$ | Mean $Xh + Nb$ |
| $e$ | `e` | $T$ | Residual $\tilde y - \mu$ (autocorrelated) |
| $\varepsilon$ | `innov` | $|\mathcal{T}_P|$ | AR innovations (white) |
| $z_k,\ r_k,\ \phi_k$ | `z`, `tanh(z)`, `phi` | $P$ | Unconstrained value, partial autocorrelation, AR coefficient |
| $\sigma$ | `sigma` | scalar | Innovation scale |
| $\nu$ | `nu` | scalar | Student-t degrees of freedom |
| $\tau_h$ | `tau_h` | scalar | Ridge scale of $h$ |
| $\tau_s$ | `tau_s` | scalar | Scale of the second-difference smoothness penalty |
| $\mathcal{T}_P$ | `t_idx` / `lag_idx` | | Time points with a full $P$-lag history inside one run, and their lags |
| $\Delta^2$ | `pt.diff(h, n=2)` | | Second difference |

## 4. Parameters

### 4.1 Learnable (sampled by NUTS)

| Parameter | Size | Prior | Role |
|---|---|---|---|
| `tau_h` | 1 | HalfNormal(1) | Learned ridge scale: how large kernel weights can be |
| `h` | $L = 20$ | $\mathcal{N}(0, \tau_h)$ | Weight of each delayed Gaussian kernel |
| `tau_s` | 1 | HalfNormal(0.5) | Learned smoothness scale (**only if `is_smooth_penalty = True`**) |
| `b` | $M$ | $\mathcal{N}(0, 1)$ | Drift (orders 0–3 per run; order 0 is the run intercept) and GSR weights |
| `z` | $P$ | $\mathcal{N}(0, 1/k)$ at lag $k$ | Unconstrained partial autocorrelations. The SD shrinks with lag (`lag_scale = pacf_scale / k`, `pacf_scale = 1`) |
| `sigma` | 1 | HalfNormal(1) | Innovation scale (standardized units) |
| `nu_minus2` | 1 | Gamma(2, 0.1) | Degrees of freedom minus 2 |

### 4.2 Deterministic (functions of the sampled parameters, stored in the trace)

| Quantity | Definition | Note |
|---|---|---|
| `phi` | Durbin–Levinson($\tanh(z)$) | AR coefficients, stationary by construction |
| `nu` | $2 + \texttt{nu\_minus2}$ | Keeps $\nu > 2$, so innovations have finite variance |
| `hrf` | $B\,h$ | HRF on the lag axis (standardized units) |

### 4.3 Extra log-density terms (`pm.Potential`)

| Name | Term | Present when |
|---|---|---|
| `smooth` | $\sum_i \log\mathcal{N}(\Delta^2 h_i \mid 0, \tau_s)$ | `is_smooth_penalty = True` |
| `lik` | $\sum_{t\in\mathcal{T}_P}\log\text{StudentT}(\varepsilon_t\mid\nu,0,\sigma)$ | always (this is the likelihood) |

### 4.4 Fixed before sampling (not learned)

| Quantity | How it is set |
|---|---|
| $P$ | `select_ar_order`: OLS fit of the full design, then AR($P$) fits to the residuals starting at `p_start = 10`. Add a lag while the last lag has $p < 0.05$, up to `pmax = 30`. Only $P$ is kept; the OLS $\phi$ values are discarded |
| Per-lag prior SD of $z$ | Constant $1/k$ |
| $X$, $N$, $B$ | Built from the stims, the data and the cedalion basis |
| $\bar y$, $s_y$ | Sample mean and SD of the concatenated $y$ |

**Note on using Potentials:** `pm.sample_prior_predictive` ignores both Potentials, and `pm.sample_posterior_predictive` / `az.loo` are not available as-is. Both would need a `pm.CustomDist(..., observed=...)` likelihood.

## 5. Step-by-step pipeline

### Step 0: Settings
`subject`, `select_parcel`, `select_chromo = 'HbO'`, `run_keys` (runs to concatenate), `is_hp_fNIRS`, `l_cutoff = 0.02` Hz, `is_smooth_penalty`, the AR-order search (`p_start`, `pmax`, `p_alpha`), `USE_GPU`, and the kernel basis `cfg_HRF_basis = {t_pre: 2 s, t_post: 18 s, t_delta: 1 s, t_std: 1 s}`.

The output name `model_name` encodes the runs, the highpass flag (`Hp`/`noHp`) and the smoothness flag (`smooth`/`noSmooth`). If `USE_GPU`, the script stops early unless JAX sees a GPU.

### Step 1: Sensitivity mask
1. Load the forward model `Adot_v26.nc`.
2. Keep brain vertices only and drop the medial wall.
3. Mark a vertex as sensitive if $\log_{10}\sum_{\text{ch}} A > -2$ at wavelength index 1.
4. `sensitive_parcels` = parcels that contain sensitive vertices. Stop if `select_parcel` is not one of them.

### Step 2: Load data and stims
- Load the stim tables from `{subject}_preprocessed_results_ar_irls_v26.pkl`.
- Load the image-space parcel time series (`parcel_ts`) from `..._IR_ts_ar_irls..._v26.pkl`.
- Match each `run_key` to an image-space run: the run's first stim onset must equal the first onset in the BIDS `events.tsv` (to within 0.01 s).

### Step 3: Per-run preprocessing (loop over `run_keys`)
1. Keep sensitive parcels and HbO, and convert to µM.
2. **Highpass** (optional): 4th-order Butterworth at 0.02 Hz, zero-phase (`sosfiltfilt`), applied before truncation.
3. **mnt-correct events**: `trial_type == 'mnt'` and `response_code == 0`.
4. **Truncate** to [first onset − 2 s, last onset + 18 s], clipped to the run, so every kernel of every kept trial fits in the window.
5. **Event design**: `glm.design_matrix.hrf_regressors(run, mnt, gauss_kernels)` builds the 20 columns of $X$ for this run (NaN → 0).
6. Reset the run's time axis to start at 0 and shift the onsets with it.

The kernel matrix $B$ and the lag axis $\tau$ come from `gauss_kernels(all_runs[0])`.

### Step 4: Full design matrix
1. `model.concatenate_runs_dms` concatenates the runs and their event designs.
2. Per-run Legendre drift, orders 0–3 (`Drift k run r`). Order 0 is the run intercept, so there is no separate global intercept.
3. Per-run global-mean regressor (`GS run r`): the mean over the sensitive parcels.
4. Each GSR column is rescaled to unit SD within its run, so the $\mathcal{N}(0,1)$ prior on $b$ is on the same footing as the Legendre columns (range [−1, 1]).
5. Split the matrix into $X$ (event columns, $T\times L$) and $N$ (nuisance columns, $T\times M$).

### Step 5: Model inputs
- $\tilde y = (y - \bar y)/s_y$.
- `run_starts`: the index where each run starts in the concatenated series. It is used so that AR lags never cross a run boundary.
- Sanity check: the number of kernels in $B$ equals the number of event columns in $X$.

### Step 6: Choose the AR order $P$
1. OLS fit of $\tilde y$ on $[X, N]$ gives residuals $\hat e$.
2. For $P = 10, 11, \dots$: fit $\hat e_t \sim \hat e_{t-1..t-P}$ by OLS, using within-run times only.
3. Stop at the first $P$ whose last-lag p-value is ≥ 0.05, or at $P = 30$ (with a warning if that lag is still significant).

### Step 7: Build the PyMC model
`build_model(y_s, X_ev, N, B, P, run_starts, is_smooth_penalty)` defines the priors in §2.5, the deterministics in §4.2, and the likelihood Potential in §2.4. The whitening step is vectorized: one gather of the $(|\mathcal{T}_P|\times P)$ lag matrix and one matrix-vector product with $\phi$.

### Step 8: Sample
NUTS, 4 chains × 500 draws after 500 tuning steps, `target_accept = 0.9`, `random_seed = 1`.
- CPU: PyMC's NUTS, 4 cores.
- GPU: numpyro NUTS, with the chains vectorized on one GPU.

### Step 9: Diagnostics
- Print the number of divergences.
- Print `az.summary` for `h`, `tau_h`, (`tau_s`), `phi`, `sigma`, `nu`: means, HDIs, $\hat R$ and ESS.

### Step 10: Posterior-mean reconstruction
All computed with the posterior means $\bar h,\ \bar b,\ \bar\phi$ (standardized units unless noted):

| Quantity | Formula |
|---|---|
| Evoked part | $X\bar h$ |
| Nuisance part | $N\bar b$ |
| Mean fit | $\mu = X\bar h + N\bar b$ |
| AR one-step prediction | $\hat e_t = \sum_k \bar\phi_k\,(\tilde y - \mu)_{t-k}$, restarted at each run |
| Full fit (µM) | $\hat y = (\mu + \hat e)\,s_y + \bar y$ |
| Event-only data (µM) | $y_{\text{partial}} = (\tilde y - N\bar b - \hat e)\,s_y$ |
| Event fit (µM) | $\hat y_{\text{event}} = X\bar h\, s_y$ |
| HRF (µM) | $\mathbb{E}[B h]\, s_y$; the script prints its peak value and latency |

### Step 11: Figures
1. **Posterior summary** (`{subject}_{parcel}_posterior.png`), 2×2 panels:
   - HRF $Bh$: posterior mean with a 95% credible band.
   - AR coefficients $\phi$: posterior mean with 95% CIs.
   - Posterior histograms of the prior scales $\tau_h$ (and $\tau_s$).
   - The data, the mean fit and the evoked fit over the first 2 minutes, with mnt-correct onsets marked.
2. **Single-event fit** (`{subject}_{parcel}_event{k}_fit.png`): −2 to 18 s around mnt-correct event `plot_event_idx`.
   - Top: $y$ vs $\hat y$.
   - Bottom: $y_{\text{partial}}$ vs $\hat y_{\text{event}}$, with other onsets in the window dotted.

### Step 12: Save
If `is_save`, write the figures to `derivatives/eeg/HRF_surf/{subject}/{model_name}/` and the posterior (`InferenceData`, netCDF) to `derivatives/eeg/{subject}/bayes/{subject}_{model_name}_{parcel}_idata.nc`.

## 6. Outputs

| Output | Content |
|---|---|
| `idata` (netCDF) | Posterior draws of all learnable and deterministic parameters (`h`, `hrf`, `b`, `z`, `phi`, `sigma`, `nu`, `tau_h`, `tau_s`) plus sampler stats |
| `*_posterior.png` | Four-panel posterior summary |
| `*_event{k}_fit.png` | Fit around one event, in µM |
| Console | AR-order search p-values, $T$, $f_s$, $L$, $M$, $P$, sampling time, divergences, `az.summary`, HRF peak in µM |

## 7. Settings reference

| Setting | Default | Effect |
|---|---|---|
| `subject` | `'sub-723'` | Subject |
| `select_parcel` | `'SomMotA_25_LH'` | Parcel to fit (must be sensitive) |
| `select_chromo` | `'HbO'` | Chromophore |
| `run_keys` | `['gradcpt1']` | Runs, concatenated in this order |
| `is_hp_fNIRS` / `l_cutoff` | `True` / 0.02 Hz | Per-run highpass before truncation |
| `is_smooth_penalty` | `True` | Adds `tau_s` and the second-difference penalty on $h$ |
| `p_start`, `pmax`, `p_alpha` | 10, 30, 0.05 | AR-order search |
| `cfg_HRF_basis` | −2 to 18 s, 1 s spacing, `t_std` 1 s | Gaussian kernel basis ($L = 20$) |
| `DRIFT_ORDER` | 3 (from `params_setting`) | Legendre orders 0–3 per run |
| `USE_GPU` | `False` | numpyro on GPU vs PyMC on CPU |
| `plot_event_idx`, `plot_win` | 2, (−2, 18) s | Event shown in the single-event figure |
| `is_save` | `True` | Save figures and posterior |